# Video Swin Transformer — Temporal Augmentation

This notebook contains the temporal-augmentation experiment for 10-class cricket shot classification using pretrained **Swin3D-T (Kinetics-400)**.

## Experiment idea
- Input: 32 frames per video, 224×224
- Training: random segment temporal sampling
- Validation/Test: deterministic uniform temporal sampling
- Spatial preprocessing remains the pretrained Video Swin preprocessing
- Full fine-tuning with AdamW and differential learning rates
- Best checkpoint selected using validation accuracy

The cleaned notebook removes inherited baseline/restart/debug cells while preserving the experiment code and recorded outputs.


## Temporal Sampling Strategy

For training, the complete video timeline is divided into **32 temporal segments** and one frame is randomly sampled from each segment. Therefore, the same training video can produce a different 32-frame clip on different accesses while preserving chronological order.

For validation and testing, 32 frames are sampled deterministically using uniform `linspace` positions. Short videos fall back to deterministic uniform sampling.

This experiment changes **temporal sampling only**; it does not add custom spatial augmentation.


In [ ]:
# ============================================================
# RESTORE DEVICE
# ============================================================

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print("Device    :", device)

if torch.cuda.is_available():
    print("GPU       :", torch.cuda.get_device_name(0))
    print("GPU count :", torch.cuda.device_count())

Device    : cuda:0
GPU       : Tesla T4
GPU count : 2


In [ ]:
# ============================================================
# FIX DATASET PATH
# ============================================================

DATASET_ROOT = (
    "/kaggle/input/datasets/nivrithareddykunduru/"
    "cricketshot-dataset/dataset"
)

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR   = os.path.join(DATASET_ROOT, "val")
TEST_DIR  = os.path.join(DATASET_ROOT, "test")


print("=" * 60)
print("DATASET PATH CHECK")
print("=" * 60)

print("Dataset root :", DATASET_ROOT)
print("Dataset      :", os.path.exists(DATASET_ROOT))
print("Train        :", os.path.exists(TRAIN_DIR))
print("Val          :", os.path.exists(VAL_DIR))
print("Test         :", os.path.exists(TEST_DIR))

print("=" * 60)

DATASET PATH CHECK
Dataset root : /kaggle/input/datasets/nivrithareddykunduru/cricketshot-dataset/dataset
Dataset      : True
Train        : True
Val          : True
Test         : True


In [ ]:
# ============================================================
# TEMPORAL AUGMENTATION
# Baseline spatial preprocessing + random temporal sampling
# ============================================================

from torchvision.models.video import Swin3D_T_Weights


# ============================================================
# 1. BASELINE SPATIAL PREPROCESSING
# ============================================================

weights = Swin3D_T_Weights.KINETICS400_V1

preprocess = weights.transforms()

train_transform = preprocess
val_transform   = preprocess
test_transform  = preprocess


# ============================================================
# 2. DETERMINISTIC SAMPLING
# Used for VALIDATION and TEST
# ============================================================

def uniform_temporal_indices(total_frames, num_frames=32):

    if total_frames <= 0:
        return np.zeros(num_frames, dtype=np.int64)

    indices = np.linspace(
        0,
        total_frames - 1,
        num_frames
    )

    return np.round(indices).astype(np.int64)


# ============================================================
# 3. RANDOM SEGMENT SAMPLING
# Used ONLY for TRAINING
# ============================================================

def random_temporal_indices(total_frames, num_frames=32):

    if total_frames <= 0:
        return np.zeros(num_frames, dtype=np.int64)

    # Short videos:
    # use deterministic positions because there aren't
    # enough unique frames for 32 temporal segments.
    if total_frames < num_frames:

        indices = np.linspace(
            0,
            total_frames - 1,
            num_frames
        )

        return np.round(indices).astype(np.int64)

    # Divide entire video timeline into 32 segments
    boundaries = np.linspace(
        0,
        total_frames,
        num_frames + 1
    )

    indices = []

    for i in range(num_frames):

        start = int(np.floor(boundaries[i]))
        end = int(np.floor(boundaries[i + 1]))

        # Ensure valid non-empty range
        end = max(end, start + 1)

        # np.random.randint upper bound is exclusive
        frame_idx = np.random.randint(
            start,
            min(end, total_frames)
        )

        indices.append(frame_idx)

    return np.array(
        indices,
        dtype=np.int64
    )


# ============================================================
# 4. QUICK SANITY CHECK
# ============================================================

example_total_frames = 64

val_indices = uniform_temporal_indices(
    example_total_frames,
    NUM_FRAMES
)

train_indices_1 = random_temporal_indices(
    example_total_frames,
    NUM_FRAMES
)

train_indices_2 = random_temporal_indices(
    example_total_frames,
    NUM_FRAMES
)


print("=" * 60)
print("TEMPORAL SAMPLING CHECK")
print("=" * 60)

print("\nValidation/Test:")
print(val_indices)

print("\nTraining sample #1:")
print(train_indices_1)

print("\nTraining sample #2:")
print(train_indices_2)

print("\nTrain samples identical:",
      np.array_equal(train_indices_1, train_indices_2))

print(
    "Train indices sorted:",
    np.all(np.diff(train_indices_1) >= 0)
)

print("\nSpatial preprocessing:")
print(preprocess)

print("=" * 60)

TEMPORAL SAMPLING CHECK

Validation/Test:
[ 0  2  4  6  8 10 12 14 16 18 20 22 24 26 28 30 33 35 37 39 41 43 45 47
 49 51 53 55 57 59 61 63]

Training sample #1:
[ 1  2  4  7  8 10 13 14 17 19 21 22 25 27 28 31 32 34 37 38 40 42 45 47
 49 51 53 54 56 59 61 62]

Training sample #2:
[ 1  3  4  6  8 10 13 15 16 19 21 22 25 27 29 30 33 34 37 39 41 42 45 47
 48 51 53 54 57 59 61 62]

Train samples identical: False
Train indices sorted: True

Spatial preprocessing:
VideoClassification(
    crop_size=[224, 224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [ ]:
# ============================================================
# CRICKET VIDEO DATASET — TEMPORAL AUGMENTATION
# ============================================================

class CricketVideoDataset(Dataset):

    def __init__(
        self,
        root_dir,
        split,
        transform,
        num_frames=32
    ):

        self.root_dir = root_dir
        self.split = split
        self.transform = transform
        self.num_frames = num_frames

        self.samples = []

        # ----------------------------------------------------
        # Collect video paths
        # ----------------------------------------------------

        for class_name in CLASSES:

            class_dir = os.path.join(
                root_dir,
                class_name
            )

            if not os.path.isdir(class_dir):
                continue

            label = CLASS_TO_IDX[class_name]

            for file_name in sorted(os.listdir(class_dir)):

                if file_name.lower().endswith(
                    (".avi", ".mp4", ".mov", ".mkv")
                ):

                    video_path = os.path.join(
                        class_dir,
                        file_name
                    )

                    self.samples.append(
                        (video_path, label)
                    )

        print(
            f"Loaded {len(self.samples)} {split} videos"
        )


    def __len__(self):

        return len(self.samples)


    def __getitem__(self, idx):

        video_path, label = self.samples[idx]

        cap = cv2.VideoCapture(video_path)

        total_frames = int(
            cap.get(cv2.CAP_PROP_FRAME_COUNT)
        )

        # ----------------------------------------------------
        # TEMPORAL SAMPLING
        # ----------------------------------------------------

        if self.split == "train":

            frame_indices = random_temporal_indices(
                total_frames,
                self.num_frames
            )

        else:

            frame_indices = uniform_temporal_indices(
                total_frames,
                self.num_frames
            )


        # ----------------------------------------------------
        # READ SELECTED FRAMES
        # ----------------------------------------------------

        frames = []

        for frame_idx in frame_indices:

            cap.set(
                cv2.CAP_PROP_POS_FRAMES,
                int(frame_idx)
            )

            success, frame = cap.read()

            if success:

                # OpenCV BGR -> RGB
                frame = cv2.cvtColor(
                    frame,
                    cv2.COLOR_BGR2RGB
                )

                frames.append(frame)


        cap.release()


        # ----------------------------------------------------
        # SAFETY — FAILED DECODING
        # ----------------------------------------------------

        if len(frames) == 0:

            raise RuntimeError(
                f"Could not decode video: {video_path}"
            )


        # If a frame failed to decode,
        # repeat the last successfully decoded frame.
        while len(frames) < self.num_frames:

            frames.append(
                frames[-1].copy()
            )


        # Ensure exactly num_frames
        frames = frames[:self.num_frames]


        # ----------------------------------------------------
        # NUMPY -> TORCH
        #
        # [T, H, W, C]
        #       ↓
        # [T, C, H, W]
        # ----------------------------------------------------

        frames = np.stack(frames)

        frames = torch.from_numpy(
            frames
        ).permute(
            0, 3, 1, 2
        )


        # ----------------------------------------------------
        # OFFICIAL KINETICS PREPROCESSING
        #
        # input : [T,C,H,W]
        # output: [C,T,H,W]
        # ----------------------------------------------------

        if self.transform is not None:

            frames = self.transform(
                frames
            )


        return frames, label


# ============================================================
# CREATE DATASETS
# ============================================================

train_dataset = CricketVideoDataset(
    TRAIN_DIR,
    split="train",
    transform=train_transform,
    num_frames=NUM_FRAMES
)

val_dataset = CricketVideoDataset(
    VAL_DIR,
    split="val",
    transform=val_transform,
    num_frames=NUM_FRAMES
)

test_dataset = CricketVideoDataset(
    TEST_DIR,
    split="test",
    transform=test_transform,
    num_frames=NUM_FRAMES
)


# ============================================================
# DATALOADERS
# ============================================================

BATCH_SIZE = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ============================================================
# CHECK
# ============================================================

print("\n" + "=" * 60)
print("TEMPORAL DATASET CHECK")
print("=" * 60)

print("Train videos :", len(train_dataset))
print("Val videos   :", len(val_dataset))
print("Test videos  :", len(test_dataset))

print()

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Test batches :", len(test_loader))

print("=" * 60)

Loaded 1321 train videos
Loaded 283 val videos
Loaded 284 test videos

TEMPORAL DATASET CHECK
Train videos : 1321
Val videos   : 283
Test videos  : 284

Train batches: 661
Val batches  : 142
Test batches : 142


In [ ]:
# ============================================================
# TEMPORAL AUGMENTATION — SANITY CHECK
# ============================================================

print("=" * 60)
print("TEMPORAL AUGMENTATION — SANITY CHECK")
print("=" * 60)


# ------------------------------------------------------------
# TRAIN — should be different
# ------------------------------------------------------------

train_video_1, train_label_1 = train_dataset[0]
train_video_2, train_label_2 = train_dataset[0]

train_difference = torch.mean(
    torch.abs(train_video_1 - train_video_2)
).item()

train_identical = torch.equal(
    train_video_1,
    train_video_2
)


# ------------------------------------------------------------
# VALIDATION — should be identical
# ------------------------------------------------------------

val_video_1, val_label_1 = val_dataset[0]
val_video_2, val_label_2 = val_dataset[0]

val_difference = torch.mean(
    torch.abs(val_video_1 - val_video_2)
).item()

val_identical = torch.equal(
    val_video_1,
    val_video_2
)


# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print("\nTRAIN")
print("Shape             :", train_video_1.shape)
print("Label             :", train_label_1)
print("Mean difference   :", train_difference)
print("Outputs identical :", train_identical)


print("\nVALIDATION")
print("Shape             :", val_video_1.shape)
print("Label             :", val_label_1)
print("Mean difference   :", val_difference)
print("Outputs identical :", val_identical)


print("\nExpected:")
print("Train identical   -> False")
print("Val identical     -> True")

print("=" * 60)

TEMPORAL AUGMENTATION — SANITY CHECK

TRAIN
Shape             : torch.Size([3, 32, 224, 224])
Label             : 0
Mean difference   : 0.027112174779176712
Outputs identical : False

VALIDATION
Shape             : torch.Size([3, 32, 224, 224])
Label             : 0
Mean difference   : 0.0
Outputs identical : True

Expected:
Train identical   -> False
Val identical     -> True


In [ ]:
# ============================================================
# RETRY KINETICS-400 PRETRAINED MODEL DOWNLOAD
# ============================================================

import time
import torch
import torch.nn as nn

from torchvision.models.video import (
    swin3d_t,
    Swin3D_T_Weights
)

weights = Swin3D_T_Weights.KINETICS400_V1

for attempt in range(1, 4):

    try:

        print(f"Attempt {attempt}/3...")

        model = swin3d_t(
            weights=weights
        )

        print("✅ Kinetics-400 pretrained Swin3D-T loaded!")
        break

    except Exception as e:

        print(
            f"❌ Attempt {attempt} failed:",
            type(e).__name__
        )

        if attempt == 3:
            raise

        print("Waiting 10 seconds before retry...\n")
        time.sleep(10)


# Replace original 400-class head
in_features = model.head.in_features

model.head = nn.Linear(
    in_features,
    NUM_CLASSES
)

model = model.to(device)

print("New classifier :", model.head)
print("Device         :", device)

Attempt 1/3...
Downloading: "https://download.pytorch.org/models/swin3d_t-7615ae03.pth" to /root/.cache/torch/hub/checkpoints/swin3d_t-7615ae03.pth


100%|██████████| 122M/122M [00:00<00:00, 207MB/s] 


✅ Kinetics-400 pretrained Swin3D-T loaded!
New classifier : Linear(in_features=768, out_features=10, bias=True)
Device         : cuda:0


In [ ]:
# ============================================================
# TEMPORAL AUGMENTATION — TRAINING + SAVE SETUP
# ============================================================

import os
import json


# ============================================================
# 1. OPTIMIZER
# Same settings as baseline
# ============================================================

BACKBONE_LR = 1e-5
HEAD_LR = 1e-4
WEIGHT_DECAY = 1e-4


backbone_params = [
    param
    for name, param in model.named_parameters()
    if not name.startswith("head.")
]


optimizer = torch.optim.AdamW(
    [
        {
            "params": backbone_params,
            "lr": BACKBONE_LR
        },
        {
            "params": model.head.parameters(),
            "lr": HEAD_LR
        }
    ],
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# 2. LOSS
# Same as baseline
# ============================================================

criterion = nn.CrossEntropyLoss()


# ============================================================
# 3. AMP
# ============================================================

scaler = torch.amp.GradScaler("cuda")


# ============================================================
# 4. SAVE DIRECTORY
# ============================================================

SAVE_DIR = "/kaggle/working/video_swin_temporal_aug"

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


LATEST_PATH = os.path.join(
    SAVE_DIR,
    "temporal_aug_latest.pth"
)

BEST_PATH = os.path.join(
    SAVE_DIR,
    "temporal_aug_best.pth"
)

HISTORY_PATH = os.path.join(
    SAVE_DIR,
    "temporal_aug_history.json"
)

CONFIG_PATH = os.path.join(
    SAVE_DIR,
    "temporal_aug_config.json"
)


# ============================================================
# 5. HISTORY
# ============================================================

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_acc = 0.0


# ============================================================
# 6. CONFIG
# ============================================================

config = {

    "experiment": "temporal_augmentation",

    "model": "swin3d_t",
    "pretrained": "KINETICS400_V1",

    "num_classes": NUM_CLASSES,
    "num_frames": NUM_FRAMES,
    "image_size": IMAGE_SIZE,

    "batch_size": BATCH_SIZE,

    "backbone_lr": BACKBONE_LR,
    "head_lr": HEAD_LR,
    "weight_decay": WEIGHT_DECAY,

    "temporal_sampling_train":
        "random_segment_sampling",

    "temporal_sampling_val_test":
        "uniform_linspace",

    "spatial_preprocessing":
        "KINETICS400_V1_default",

    "seed": SEED,

    "classes": CLASSES
}


with open(CONFIG_PATH, "w") as f:
    json.dump(
        config,
        f,
        indent=4
    )


# ============================================================
# 7. VERIFY PARAMETER GROUPS
# ============================================================

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print("=" * 60)
print("TEMPORAL AUGMENTATION — TRAINING SETUP")
print("=" * 60)

print(f"Total parameters : {total_params:,}")
print(f"Trainable params : {trainable_params:,}")

print()

print(
    "Backbone LR      :",
    optimizer.param_groups[0]["lr"]
)

print(
    "Head LR          :",
    optimizer.param_groups[1]["lr"]
)

print("Weight decay     :", WEIGHT_DECAY)
print("Loss             : CrossEntropyLoss")
print("Label smoothing  : NO")
print("AMP              : Enabled")

print()

print("Save directory   :", SAVE_DIR)
print("Latest checkpoint:", LATEST_PATH)
print("Best checkpoint  :", BEST_PATH)

print("=" * 60)

TEMPORAL AUGMENTATION — TRAINING SETUP
Total parameters : 27,858,160
Trainable params : 27,858,160

Backbone LR      : 1e-05
Head LR          : 0.0001
Weight decay     : 0.0001
Loss             : CrossEntropyLoss
Label smoothing  : NO
AMP              : Enabled

Save directory   : /kaggle/working/video_swin_temporal_aug
Latest checkpoint: /kaggle/working/video_swin_temporal_aug/temporal_aug_latest.pth
Best checkpoint  : /kaggle/working/video_swin_temporal_aug/temporal_aug_best.pth


In [ ]:
# ============================================================
# TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler,
    device
):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (videos, labels) in enumerate(loader):

        videos = videos.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            outputs = model(videos)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()

        scaler.step(optimizer)
        scaler.update()

        running_loss += (
            loss.item() * videos.size(0)
        )

        predictions = outputs.argmax(dim=1)

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)


        if (batch_idx + 1) % 50 == 0:

            print(
                f"Batch {batch_idx + 1}/{len(loader)} "
                f"| Loss: {loss.item():.4f}"
            )


    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


# ============================================================
# VALIDATION
# ============================================================

@torch.no_grad()
def validate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for videos, labels in loader:

        videos = videos.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            outputs = model(videos)
            loss = criterion(outputs, labels)

        running_loss += (
            loss.item() * videos.size(0)
        )

        predictions = outputs.argmax(dim=1)

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)


    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


# ============================================================
# SAVE TRAINING STATE
# ============================================================

def save_training_state(
    epoch,
    model,
    optimizer,
    scaler,
    best_val_acc,
    history,
    config,
    is_best=False
):

    checkpoint = {

        "epoch": epoch,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scaler_state_dict":
            scaler.state_dict(),

        "best_val_acc":
            best_val_acc,

        "history":
            history,

        "config":
            config,

        "classes":
            CLASSES
    }


    # Always save latest
    torch.save(
        checkpoint,
        LATEST_PATH
    )


    # Save separately when validation improves
    if is_best:

        torch.save(
            checkpoint,
            BEST_PATH
        )


    # Save history separately
    with open(HISTORY_PATH, "w") as f:

        json.dump(
            history,
            f,
            indent=4
        )


    # Save config separately
    with open(CONFIG_PATH, "w") as f:

        json.dump(
            config,
            f,
            indent=4
        )


print("✅ train_one_epoch ready")
print("✅ validate ready")
print("✅ checkpoint saving ready")

✅ train_one_epoch ready
✅ validate ready
✅ checkpoint saving ready


In [ ]:
# ============================================================
# TEMPORAL AUGMENTATION — EPOCH 1
# ============================================================

import time

epoch = 1

print("\n" + "=" * 60)
print(f"TEMPORAL AUGMENTATION — EPOCH {epoch}")
print("=" * 60)

start_time = time.time()


# ============================================================
# TRAIN
# ============================================================

train_loss, train_acc = train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    scaler,
    device
)


# ============================================================
# VALIDATE
# ============================================================

val_loss, val_acc = validate(
    model,
    val_loader,
    criterion,
    device
)


# ============================================================
# UPDATE HISTORY
# ============================================================

history["train_loss"].append(train_loss)
history["train_acc"].append(train_acc)

history["val_loss"].append(val_loss)
history["val_acc"].append(val_acc)


# ============================================================
# CHECK BEST
# ============================================================

is_best = val_acc > best_val_acc

if is_best:

    best_val_acc = val_acc

    print("\n⭐ New best checkpoint saved")


# ============================================================
# SAVE CHECKPOINT
# ============================================================

save_training_state(
    epoch,
    model,
    optimizer,
    scaler,
    best_val_acc,
    history,
    config,
    is_best=is_best
)


# ============================================================
# RESULTS
# ============================================================

elapsed_minutes = (
    time.time() - start_time
) / 60


print("\n" + "=" * 60)

print(f"Train Loss : {train_loss:.4f}")
print(f"Train Acc  : {train_acc * 100:.2f}%")

print(f"Val Loss   : {val_loss:.4f}")
print(f"Val Acc    : {val_acc * 100:.2f}%")

print(f"Best Val   : {best_val_acc * 100:.2f}%")

print(
    f"LR backbone: "
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    f"LR head    : "
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

print(f"Time       : {elapsed_minutes:.2f} min")

print("=" * 60)


TEMPORAL AUGMENTATION — EPOCH 1
Batch 50/661 | Loss: 2.3721
Batch 100/661 | Loss: 2.0781
Batch 150/661 | Loss: 2.6943
Batch 200/661 | Loss: 2.3672
Batch 250/661 | Loss: 1.8643
Batch 300/661 | Loss: 2.7988
Batch 350/661 | Loss: 1.4315
Batch 400/661 | Loss: 1.2561
Batch 450/661 | Loss: 1.9392
Batch 500/661 | Loss: 1.2554
Batch 550/661 | Loss: 0.6949
Batch 600/661 | Loss: 1.0627
Batch 650/661 | Loss: 1.2938

⭐ New best checkpoint saved

Train Loss : 1.7163
Train Acc  : 40.42%
Val Loss   : 1.1819
Val Acc    : 56.89%
Best Val   : 56.89%
LR backbone: 1.00e-05
LR head    : 1.00e-04
Time       : 21.40 min


In [ ]:
# ============================================================
# TEMPORAL AUGMENTATION — EPOCH 2
# ============================================================

epoch = 2

print("\n" + "=" * 60)
print(f"TEMPORAL AUGMENTATION — EPOCH {epoch}")
print("=" * 60)

start_time = time.time()


# TRAIN
train_loss, train_acc = train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    scaler,
    device
)


# VALIDATE
val_loss, val_acc = validate(
    model,
    val_loader,
    criterion,
    device
)


# UPDATE HISTORY
history["train_loss"].append(train_loss)
history["train_acc"].append(train_acc)
history["val_loss"].append(val_loss)
history["val_acc"].append(val_acc)


# CHECK BEST
is_best = val_acc > best_val_acc

if is_best:
    best_val_acc = val_acc
    print("\n⭐ New best checkpoint saved")


# SAVE
save_training_state(
    epoch,
    model,
    optimizer,
    scaler,
    best_val_acc,
    history,
    config,
    is_best=is_best
)


# RESULTS
elapsed_minutes = (time.time() - start_time) / 60

print("\n" + "=" * 60)

print(f"Train Loss : {train_loss:.4f}")
print(f"Train Acc  : {train_acc * 100:.2f}%")

print(f"Val Loss   : {val_loss:.4f}")
print(f"Val Acc    : {val_acc * 100:.2f}%")

print(f"Best Val   : {best_val_acc * 100:.2f}%")

print(
    f"LR backbone: "
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    f"LR head    : "
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

print(f"Time       : {elapsed_minutes:.2f} min")

print("=" * 60)


TEMPORAL AUGMENTATION — EPOCH 2
Batch 50/661 | Loss: 1.7007
Batch 100/661 | Loss: 0.4083
Batch 150/661 | Loss: 2.3867
Batch 200/661 | Loss: 0.7560
Batch 250/661 | Loss: 0.0879
Batch 300/661 | Loss: 0.4247
Batch 350/661 | Loss: 0.2100
Batch 400/661 | Loss: 0.4935
Batch 450/661 | Loss: 0.7194
Batch 500/661 | Loss: 0.5894
Batch 550/661 | Loss: 0.2536
Batch 600/661 | Loss: 0.2146
Batch 650/661 | Loss: 2.2397

⭐ New best checkpoint saved

Train Loss : 0.8072
Train Acc  : 74.34%
Val Loss   : 0.7263
Val Acc    : 74.56%
Best Val   : 74.56%
LR backbone: 1.00e-05
LR head    : 1.00e-04
Time       : 20.63 min


In [ ]:
# ============================================================
# TEMPORAL AUGMENTATION — EPOCH 3
# ============================================================

epoch = 3

print("\n" + "=" * 60)
print(f"TEMPORAL AUGMENTATION — EPOCH {epoch}")
print("=" * 60)

start_time = time.time()


train_loss, train_acc = train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    scaler,
    device
)


val_loss, val_acc = validate(
    model,
    val_loader,
    criterion,
    device
)


history["train_loss"].append(train_loss)
history["train_acc"].append(train_acc)
history["val_loss"].append(val_loss)
history["val_acc"].append(val_acc)


is_best = val_acc > best_val_acc

if is_best:
    best_val_acc = val_acc
    print("\n⭐ New best checkpoint saved")


save_training_state(
    epoch,
    model,
    optimizer,
    scaler,
    best_val_acc,
    history,
    config,
    is_best=is_best
)


elapsed_minutes = (time.time() - start_time) / 60


print("\n" + "=" * 60)

print(f"Train Loss : {train_loss:.4f}")
print(f"Train Acc  : {train_acc * 100:.2f}%")

print(f"Val Loss   : {val_loss:.4f}")
print(f"Val Acc    : {val_acc * 100:.2f}%")

print(f"Best Val   : {best_val_acc * 100:.2f}%")

print(
    f"LR backbone: "
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    f"LR head    : "
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

print(f"Time       : {elapsed_minutes:.2f} min")

print("=" * 60)


TEMPORAL AUGMENTATION — EPOCH 3
Batch 50/661 | Loss: 0.2242
Batch 100/661 | Loss: 0.9863
Batch 150/661 | Loss: 0.3575
Batch 200/661 | Loss: 0.5349
Batch 250/661 | Loss: 0.5408
Batch 300/661 | Loss: 0.6815
Batch 350/661 | Loss: 0.4388
Batch 400/661 | Loss: 0.2342
Batch 450/661 | Loss: 0.1771
Batch 500/661 | Loss: 0.9719
Batch 550/661 | Loss: 0.2076
Batch 600/661 | Loss: 0.5770
Batch 650/661 | Loss: 0.0095

⭐ New best checkpoint saved

Train Loss : 0.4328
Train Acc  : 86.37%
Val Loss   : 0.5246
Val Acc    : 80.21%
Best Val   : 80.21%
LR backbone: 1.00e-05
LR head    : 1.00e-04
Time       : 20.49 min


In [ ]:
# ============================================================
# TEMPORAL AUGMENTATION — EPOCH 4
# ============================================================

epoch = 4

print("\n" + "=" * 60)
print(f"TEMPORAL AUGMENTATION — EPOCH {epoch}")
print("=" * 60)

start_time = time.time()


# TRAIN
train_loss, train_acc = train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    scaler,
    device
)


# VALIDATE
val_loss, val_acc = validate(
    model,
    val_loader,
    criterion,
    device
)


# UPDATE HISTORY
history["train_loss"].append(train_loss)
history["train_acc"].append(train_acc)
history["val_loss"].append(val_loss)
history["val_acc"].append(val_acc)


# CHECK BEST
is_best = val_acc > best_val_acc

if is_best:
    best_val_acc = val_acc
    print("\n⭐ New best checkpoint saved")


# SAVE
save_training_state(
    epoch,
    model,
    optimizer,
    scaler,
    best_val_acc,
    history,
    config,
    is_best=is_best
)


# RESULTS
elapsed_minutes = (time.time() - start_time) / 60

print("\n" + "=" * 60)

print(f"Train Loss : {train_loss:.4f}")
print(f"Train Acc  : {train_acc * 100:.2f}%")

print(f"Val Loss   : {val_loss:.4f}")
print(f"Val Acc    : {val_acc * 100:.2f}%")

print(f"Best Val   : {best_val_acc * 100:.2f}%")

print(
    f"LR backbone: "
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    f"LR head    : "
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

print(f"Time       : {elapsed_minutes:.2f} min")

print("=" * 60)


TEMPORAL AUGMENTATION — EPOCH 4
Batch 50/661 | Loss: 0.2763
Batch 100/661 | Loss: 0.0494
Batch 150/661 | Loss: 0.2088
Batch 200/661 | Loss: 0.0103
Batch 250/661 | Loss: 0.8063
Batch 300/661 | Loss: 0.0535
Batch 350/661 | Loss: 0.2036
Batch 400/661 | Loss: 0.1202
Batch 450/661 | Loss: 0.0749
Batch 500/661 | Loss: 0.2770
Batch 550/661 | Loss: 0.0037
Batch 600/661 | Loss: 1.0549
Batch 650/661 | Loss: 0.0399

⭐ New best checkpoint saved

Train Loss : 0.2847
Train Acc  : 92.58%
Val Loss   : 0.5198
Val Acc    : 83.75%
Best Val   : 83.75%
LR backbone: 1.00e-05
LR head    : 1.00e-04
Time       : 20.66 min


In [ ]:
# ============================================================
# TEMPORAL AUGMENTATION — EPOCH 5
# ============================================================

epoch = 5

print("\n" + "=" * 60)
print(f"TEMPORAL AUGMENTATION — EPOCH {epoch}")
print("=" * 60)

start_time = time.time()


train_loss, train_acc = train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    scaler,
    device
)


val_loss, val_acc = validate(
    model,
    val_loader,
    criterion,
    device
)


history["train_loss"].append(train_loss)
history["train_acc"].append(train_acc)
history["val_loss"].append(val_loss)
history["val_acc"].append(val_acc)


is_best = val_acc > best_val_acc

if is_best:
    best_val_acc = val_acc
    print("\n⭐ New best checkpoint saved")


save_training_state(
    epoch,
    model,
    optimizer,
    scaler,
    best_val_acc,
    history,
    config,
    is_best=is_best
)


elapsed_minutes = (time.time() - start_time) / 60


print("\n" + "=" * 60)

print(f"Train Loss : {train_loss:.4f}")
print(f"Train Acc  : {train_acc * 100:.2f}%")

print(f"Val Loss   : {val_loss:.4f}")
print(f"Val Acc    : {val_acc * 100:.2f}%")

print(f"Best Val   : {best_val_acc * 100:.2f}%")

print(
    f"LR backbone: "
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    f"LR head    : "
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

print(f"Time       : {elapsed_minutes:.2f} min")

print("=" * 60)


TEMPORAL AUGMENTATION — EPOCH 5
Batch 50/661 | Loss: 0.0092
Batch 100/661 | Loss: 0.0105
Batch 150/661 | Loss: 0.3928
Batch 200/661 | Loss: 0.0604
Batch 250/661 | Loss: 0.0548
Batch 300/661 | Loss: 0.0007
Batch 350/661 | Loss: 0.0079
Batch 400/661 | Loss: 0.0584
Batch 450/661 | Loss: 0.3342
Batch 500/661 | Loss: 0.0421
Batch 550/661 | Loss: 0.2723
Batch 600/661 | Loss: 0.0123
Batch 650/661 | Loss: 0.0428

⭐ New best checkpoint saved

Train Loss : 0.1771
Train Acc  : 95.53%
Val Loss   : 0.4200
Val Acc    : 85.87%
Best Val   : 85.87%
LR backbone: 1.00e-05
LR head    : 1.00e-04
Time       : 20.56 min


In [ ]:
# ============================================================
# TEMPORAL AUGMENTATION — EPOCH 6
# ============================================================

epoch = 6

print("\n" + "=" * 60)
print(f"TEMPORAL AUGMENTATION — EPOCH {epoch}")
print("=" * 60)

start_time = time.time()


train_loss, train_acc = train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    scaler,
    device
)

val_loss, val_acc = validate(
    model,
    val_loader,
    criterion,
    device
)


history["train_loss"].append(train_loss)
history["train_acc"].append(train_acc)
history["val_loss"].append(val_loss)
history["val_acc"].append(val_acc)


is_best = val_acc > best_val_acc

if is_best:
    best_val_acc = val_acc
    print("\n⭐ New best checkpoint saved")


save_training_state(
    epoch,
    model,
    optimizer,
    scaler,
    best_val_acc,
    history,
    config,
    is_best=is_best
)


elapsed_minutes = (time.time() - start_time) / 60


print("\n" + "=" * 60)

print(f"Train Loss : {train_loss:.4f}")
print(f"Train Acc  : {train_acc * 100:.2f}%")

print(f"Val Loss   : {val_loss:.4f}")
print(f"Val Acc    : {val_acc * 100:.2f}%")

print(f"Best Val   : {best_val_acc * 100:.2f}%")

print(
    f"LR backbone: "
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    f"LR head    : "
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

print(f"Time       : {elapsed_minutes:.2f} min")

print("=" * 60)


TEMPORAL AUGMENTATION — EPOCH 6
Batch 50/661 | Loss: 0.1998
Batch 100/661 | Loss: 0.0492
Batch 150/661 | Loss: 0.0052
Batch 200/661 | Loss: 0.2046
Batch 250/661 | Loss: 0.2066
Batch 300/661 | Loss: 0.0016
Batch 350/661 | Loss: 0.0689
Batch 400/661 | Loss: 0.0762
Batch 450/661 | Loss: 0.1024
Batch 500/661 | Loss: 0.0140
Batch 550/661 | Loss: 0.1139
Batch 600/661 | Loss: 0.0396
Batch 650/661 | Loss: 0.0544

⭐ New best checkpoint saved

Train Loss : 0.1120
Train Acc  : 97.12%
Val Loss   : 0.4087
Val Acc    : 86.57%
Best Val   : 86.57%
LR backbone: 1.00e-05
LR head    : 1.00e-04
Time       : 21.04 min


In [ ]:
# ============================================================
# TEMPORAL AUGMENTATION — EPOCH 7
# ============================================================

epoch = 7

print("\n" + "=" * 60)
print(f"TEMPORAL AUGMENTATION — EPOCH {epoch}")
print("=" * 60)

start_time = time.time()


train_loss, train_acc = train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    scaler,
    device
)

val_loss, val_acc = validate(
    model,
    val_loader,
    criterion,
    device
)


history["train_loss"].append(train_loss)
history["train_acc"].append(train_acc)
history["val_loss"].append(val_loss)
history["val_acc"].append(val_acc)


is_best = val_acc > best_val_acc

if is_best:
    best_val_acc = val_acc
    print("\n⭐ New best checkpoint saved")


save_training_state(
    epoch,
    model,
    optimizer,
    scaler,
    best_val_acc,
    history,
    config,
    is_best=is_best
)


elapsed_minutes = (time.time() - start_time) / 60


print("\n" + "=" * 60)

print(f"Train Loss : {train_loss:.4f}")
print(f"Train Acc  : {train_acc * 100:.2f}%")

print(f"Val Loss   : {val_loss:.4f}")
print(f"Val Acc    : {val_acc * 100:.2f}%")

print(f"Best Val   : {best_val_acc * 100:.2f}%")

print(
    f"LR backbone: "
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    f"LR head    : "
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

print(f"Time       : {elapsed_minutes:.2f} min")

print("=" * 60)


TEMPORAL AUGMENTATION — EPOCH 7
Batch 50/661 | Loss: 1.9233
Batch 100/661 | Loss: 0.0014
Batch 150/661 | Loss: 0.4149
Batch 200/661 | Loss: 0.0094
Batch 250/661 | Loss: 0.0006
Batch 300/661 | Loss: 0.0159
Batch 350/661 | Loss: 0.0026
Batch 400/661 | Loss: 0.0007
Batch 450/661 | Loss: 0.6756
Batch 500/661 | Loss: 0.0445
Batch 550/661 | Loss: 0.0087
Batch 600/661 | Loss: 0.0069
Batch 650/661 | Loss: 0.0047

Train Loss : 0.1033
Train Acc  : 96.82%
Val Loss   : 0.7537
Val Acc    : 80.57%
Best Val   : 86.57%
LR backbone: 1.00e-05
LR head    : 1.00e-04
Time       : 21.02 min


In [ ]:
# ============================================================
# REDUCE LR BEFORE EPOCH 8
# ============================================================

optimizer.param_groups[0]["lr"] = 5e-6
optimizer.param_groups[1]["lr"] = 5e-5

print("Backbone LR:", optimizer.param_groups[0]["lr"])
print("Head LR    :", optimizer.param_groups[1]["lr"])

Backbone LR: 5e-06
Head LR    : 5e-05


In [ ]:
# ============================================================
# TEMPORAL AUGMENTATION — EPOCH 8
# ============================================================

epoch = 8

print("\n" + "=" * 60)
print(f"TEMPORAL AUGMENTATION — EPOCH {epoch}")
print("=" * 60)

start_time = time.time()


train_loss, train_acc = train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    scaler,
    device
)

val_loss, val_acc = validate(
    model,
    val_loader,
    criterion,
    device
)


history["train_loss"].append(train_loss)
history["train_acc"].append(train_acc)
history["val_loss"].append(val_loss)
history["val_acc"].append(val_acc)


is_best = val_acc > best_val_acc

if is_best:
    best_val_acc = val_acc
    print("\n⭐ New best checkpoint saved")


save_training_state(
    epoch,
    model,
    optimizer,
    scaler,
    best_val_acc,
    history,
    config,
    is_best=is_best
)


elapsed_minutes = (time.time() - start_time) / 60


print("\n" + "=" * 60)

print(f"Train Loss : {train_loss:.4f}")
print(f"Train Acc  : {train_acc * 100:.2f}%")

print(f"Val Loss   : {val_loss:.4f}")
print(f"Val Acc    : {val_acc * 100:.2f}%")

print(f"Best Val   : {best_val_acc * 100:.2f}%")

print(
    f"LR backbone: "
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    f"LR head    : "
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

print(f"Time       : {elapsed_minutes:.2f} min")

print("=" * 60)


TEMPORAL AUGMENTATION — EPOCH 8
Batch 50/661 | Loss: 0.0119
Batch 100/661 | Loss: 0.0415
Batch 150/661 | Loss: 0.3614
Batch 200/661 | Loss: 0.0023
Batch 250/661 | Loss: 0.1647
Batch 300/661 | Loss: 0.0035
Batch 350/661 | Loss: 0.2311
Batch 400/661 | Loss: 0.1293
Batch 450/661 | Loss: 0.0006
Batch 500/661 | Loss: 0.0001
Batch 550/661 | Loss: 0.0095
Batch 600/661 | Loss: 0.0032
Batch 650/661 | Loss: 0.0062

⭐ New best checkpoint saved

Train Loss : 0.0709
Train Acc  : 98.56%
Val Loss   : 0.3673
Val Acc    : 88.69%
Best Val   : 88.69%
LR backbone: 5.00e-06
LR head    : 5.00e-05
Time       : 20.74 min


In [ ]:
# ============================================================
# TEMPORAL AUGMENTATION — EPOCH 9
# ============================================================

epoch = 9

print("\n" + "=" * 60)
print(f"TEMPORAL AUGMENTATION — EPOCH {epoch}")
print("=" * 60)

# Verify learning rates
print(
    f"Starting LR backbone: "
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    f"Starting LR head    : "
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

start_time = time.time()


# ---------------- TRAIN ----------------

train_loss, train_acc = train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    scaler,
    device
)


# ---------------- VALIDATE ----------------

val_loss, val_acc = validate(
    model,
    val_loader,
    criterion,
    device
)


# ---------------- HISTORY ----------------

history["train_loss"].append(train_loss)
history["train_acc"].append(train_acc)
history["val_loss"].append(val_loss)
history["val_acc"].append(val_acc)


# ---------------- BEST MODEL ----------------

is_best = val_acc > best_val_acc

if is_best:
    best_val_acc = val_acc
    print("\n⭐ New best checkpoint saved")


# ---------------- SAVE ----------------

save_training_state(
    epoch,
    model,
    optimizer,
    scaler,
    best_val_acc,
    history,
    config,
    is_best=is_best
)


elapsed_minutes = (time.time() - start_time) / 60


# ---------------- RESULTS ----------------

print("\n" + "=" * 60)

print(f"Train Loss : {train_loss:.4f}")
print(f"Train Acc  : {train_acc * 100:.2f}%")

print(f"Val Loss   : {val_loss:.4f}")
print(f"Val Acc    : {val_acc * 100:.2f}%")

print(f"Best Val   : {best_val_acc * 100:.2f}%")

print(
    f"LR backbone: "
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    f"LR head    : "
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

print(f"Time       : {elapsed_minutes:.2f} min")

print("=" * 60)


TEMPORAL AUGMENTATION — EPOCH 9
Starting LR backbone: 5.00e-06
Starting LR head    : 5.00e-05
Batch 50/661 | Loss: 0.0048
Batch 100/661 | Loss: 0.0003
Batch 150/661 | Loss: 0.0432
Batch 200/661 | Loss: 0.0087
Batch 250/661 | Loss: 0.0013
Batch 300/661 | Loss: 0.0115
Batch 350/661 | Loss: 0.1223
Batch 400/661 | Loss: 0.0183
Batch 450/661 | Loss: 0.0104
Batch 500/661 | Loss: 0.2386
Batch 550/661 | Loss: 0.1517
Batch 600/661 | Loss: 0.0006
Batch 650/661 | Loss: 0.0033

⭐ New best checkpoint saved

Train Loss : 0.0440
Train Acc  : 99.62%
Val Loss   : 0.3642
Val Acc    : 89.05%
Best Val   : 89.05%
LR backbone: 5.00e-06
LR head    : 5.00e-05
Time       : 21.10 min


In [ ]:
# ============================================================
# TEMPORAL AUGMENTATION — EPOCH 10
# ============================================================

epoch = 10

print("\n" + "=" * 60)
print(f"TEMPORAL AUGMENTATION — EPOCH {epoch}")
print("=" * 60)

print(
    f"Starting LR backbone: "
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    f"Starting LR head    : "
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

start_time = time.time()


# ---------------- TRAIN ----------------

train_loss, train_acc = train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    scaler,
    device
)


# ---------------- VALIDATE ----------------

val_loss, val_acc = validate(
    model,
    val_loader,
    criterion,
    device
)


# ---------------- HISTORY ----------------

history["train_loss"].append(train_loss)
history["train_acc"].append(train_acc)
history["val_loss"].append(val_loss)
history["val_acc"].append(val_acc)


# ---------------- BEST ----------------

is_best = val_acc > best_val_acc

if is_best:
    best_val_acc = val_acc
    print("\n⭐ New best checkpoint saved")


# ---------------- SAVE ----------------

save_training_state(
    epoch,
    model,
    optimizer,
    scaler,
    best_val_acc,
    history,
    config,
    is_best=is_best
)


elapsed_minutes = (time.time() - start_time) / 60


# ---------------- RESULTS ----------------

print("\n" + "=" * 60)

print(f"Train Loss : {train_loss:.4f}")
print(f"Train Acc  : {train_acc * 100:.2f}%")

print(f"Val Loss   : {val_loss:.4f}")
print(f"Val Acc    : {val_acc * 100:.2f}%")

print(f"Best Val   : {best_val_acc * 100:.2f}%")

print(
    f"LR backbone: "
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    f"LR head    : "
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

print(f"Time       : {elapsed_minutes:.2f} min")

print("=" * 60)


TEMPORAL AUGMENTATION — EPOCH 10
Starting LR backbone: 5.00e-06
Starting LR head    : 5.00e-05
Batch 50/661 | Loss: 0.1144
Batch 100/661 | Loss: 0.0055
Batch 150/661 | Loss: 0.0028
Batch 200/661 | Loss: 0.0005
Batch 250/661 | Loss: 0.0042
Batch 300/661 | Loss: 0.0067
Batch 350/661 | Loss: 0.0308
Batch 400/661 | Loss: 0.0737
Batch 450/661 | Loss: 0.0046
Batch 500/661 | Loss: 0.0019
Batch 550/661 | Loss: 0.0163
Batch 600/661 | Loss: 0.0104
Batch 650/661 | Loss: 0.0035

Train Loss : 0.0404
Train Acc  : 99.17%
Val Loss   : 0.4932
Val Acc    : 85.51%
Best Val   : 89.05%
LR backbone: 5.00e-06
LR head    : 5.00e-05
Time       : 22.01 min


In [ ]:
import os
import json
import torch

BEST_PATH = "/kaggle/working/video_swin_temporal_aug/temporal_aug_best.pth"

FINAL_MODEL_PATH = (
    "/kaggle/working/video_swin_temporal_aug/"
    "temporal_aug_final_model.pth"
)

FINAL_SUMMARY_PATH = (
    "/kaggle/working/video_swin_temporal_aug/"
    "temporal_aug_final_summary.json"
)


# Load best checkpoint
best_checkpoint = torch.load(
    BEST_PATH,
    map_location="cpu",
    weights_only=False
)

print("Selected epoch:", best_checkpoint["epoch"])
print(
    "Best validation accuracy:",
    f"{best_checkpoint['best_val_acc'] * 100:.2f}%"
)


# Save clean model weights
torch.save(
    best_checkpoint["model_state_dict"],
    FINAL_MODEL_PATH
)


# Save experiment summary
summary = {
    "experiment": "temporal_augmentation",
    "model": "swin3d_t",
    "pretrained": "KINETICS400_V1",

    "selected_epoch": best_checkpoint["epoch"],
    "best_val_accuracy": best_checkpoint["best_val_acc"],

    "baseline_best_val_accuracy": 0.8728,
    "val_improvement_percentage_points":
        (best_checkpoint["best_val_acc"] - 0.8728) * 100,

    "temporal_sampling_train": "random_segment_sampling",
    "temporal_sampling_val_test": "uniform_linspace",

    "num_frames": 32,
    "image_size": 224,

    "classes": CLASSES
}

with open(FINAL_SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=4)


print("\nSaved:")
print(FINAL_MODEL_PATH)
print(FINAL_SUMMARY_PATH)

print(
    "\nFinal model size:",
    f"{os.path.getsize(FINAL_MODEL_PATH)/(1024**2):.2f} MB"
)

Selected epoch: 9
Best validation accuracy: 89.05%

Saved:
/kaggle/working/video_swin_temporal_aug/temporal_aug_final_model.pth
/kaggle/working/video_swin_temporal_aug/temporal_aug_final_summary.json

Final model size: 120.40 MB


In [ ]:
# ============================================================
# FINAL TEST — TEMPORAL AUGMENTATION BEST CHECKPOINT
# ============================================================

import torch
import numpy as np

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)


# ------------------------------------------------------------
# 1. Load BEST temporal checkpoint
# ------------------------------------------------------------

BEST_PATH = (
    "/kaggle/working/video_swin_temporal_aug/"
    "temporal_aug_best.pth"
)

checkpoint = torch.load(
    BEST_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)
model.eval()


print("=" * 60)
print("FINAL TEST — TEMPORAL AUGMENTATION")
print("=" * 60)

print(
    "Selected best epoch:",
    checkpoint["epoch"]
)

print(
    f"Best validation accuracy: "
    f"{checkpoint['best_val_acc'] * 100:.2f}%"
)


# ------------------------------------------------------------
# 2. Test
# ------------------------------------------------------------

test_running_loss = 0.0
test_correct = 0
test_total = 0

all_labels = []
all_predictions = []


with torch.no_grad():

    for videos, labels in test_loader:

        videos = videos.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            outputs = model(videos)

            loss = criterion(
                outputs,
                labels
            )

        predictions = outputs.argmax(dim=1)


        # Loss
        test_running_loss += (
            loss.item() * labels.size(0)
        )


        # Accuracy
        test_correct += (
            predictions == labels
        ).sum().item()

        test_total += labels.size(0)


        # Store predictions
        all_labels.extend(
            labels.cpu().numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )


# ------------------------------------------------------------
# 3. Final metrics
# ------------------------------------------------------------

test_loss = test_running_loss / test_total

test_accuracy = (
    test_correct / test_total
)


print("\n" + "=" * 60)

print(
    f"Test Loss : "
    f"{test_loss:.4f}"
)

print(
    f"Test Acc  : "
    f"{test_accuracy * 100:.2f}%"
)

print(
    f"Correct   : "
    f"{test_correct}"
)

print(
    f"Wrong     : "
    f"{test_total - test_correct}"
)

print(
    f"Total     : "
    f"{test_total}"
)

print("=" * 60)


# ------------------------------------------------------------
# 4. Classification report
# ------------------------------------------------------------

print("\nCLASSIFICATION REPORT\n")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=CLASSES,
        digits=4
    )
)


# ------------------------------------------------------------
# 5. Confusion matrix
# ------------------------------------------------------------

print("\nCONFUSION MATRIX\n")

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print(cm)

FINAL TEST — TEMPORAL AUGMENTATION
Selected best epoch: 9
Best validation accuracy: 89.05%

Test Loss : 0.3736
Test Acc  : 89.44%
Correct   : 254
Wrong     : 30
Total     : 284

CLASSIFICATION REPORT

              precision    recall  f1-score   support

       cover     0.8148    0.7857    0.8000        28
     defense     0.9667    1.0000    0.9831        29
       flick     0.9583    0.8519    0.9020        27
        hook     0.8966    0.9630    0.9286        27
    late_cut     0.8889    0.8571    0.8727        28
      lofted     0.9333    0.9333    0.9333        30
        pull     0.9200    0.8519    0.8846        27
  square_cut     0.7419    0.7667    0.7541        30
    straight     0.9667    1.0000    0.9831        29
       sweep     0.8710    0.9310    0.9000        29

    accuracy                         0.8944       284
   macro avg     0.8958    0.8941    0.8941       284
weighted avg     0.8951    0.8944    0.8940       284


CONFUSION MATRIX

[[22  0  1  0  1  0  

## Final Experiment Summary

- **Total training epochs:** 10
- Learning rates were reduced before Epoch 8:
  - backbone: `1e-5 → 5e-6`
  - head: `1e-4 → 5e-5`
- **Selected best epoch:** 9
- **Best validation accuracy:** 89.05%
- **Final test loss:** 0.3736
- **Final test accuracy:** 89.44%
- **Correct predictions:** 254 / 284
- **Wrong predictions:** 30 / 284
- **Macro F1:** 0.8941
- **Weighted F1:** 0.8940

An intermediate test was performed after Epoch 8, but it is not the final reported result because training continued and Epoch 9 achieved a higher validation accuracy.
